In [1]:
import json
import cv2
import numpy as np
from tensorflow.keras.models import load_model

I0000 00:00:1775631991.799535   53808 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775631991.854051   53808 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775631992.904364   53808 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# ─── Load model and class names ───────────────────────────────────────────────
rt_model = load_model('best_asl_model.h5')

with open('class_names.json', 'r') as f:
    rt_class_names = json.load(f)

print(f"✅ Model loaded. Classes: {rt_class_names}")

# ─── Configuration ────────────────────────────────────────────────────────────
IMG_SIZE_RT = (64, 64)
CONFIDENCE_THRESHOLD = 0.7

# ROI settings
ROI_SIZE = 300

# Prediction smoothing buffer
recent_preds = []

# Initialize variables (FIX for NameError)
top_label = ""
confidence = 0.0
predictions = np.zeros(len(rt_class_names))
top3_idx = [0, 1, 2]

E0000 00:00:1775632005.788603   53808 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


✅ Model loaded. Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [3]:
# ─── Open Webcam ──────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Error: Cannot open webcam.")
    exit()

# Set camera resolution (helps window size issues)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 960)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print("🎥 Webcam opened. Place hand in the GREEN BOX.")
print("Press 'Q' to quit | Press 'S' to save screenshot")

frame_count = 0

# Create resizable window (IMPORTANT FIX)
cv2.namedWindow("ASL Detector", cv2.WINDOW_NORMAL)
cv2.resizeWindow("ASL Detector", 1000, 700)

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Failed to read frame.")
        break

    frame = cv2.flip(frame, 1)
    frame_count += 1

    h, w, _ = frame.shape

    # Dynamically position ROI (center-left)
    ROI_X = int(w * 0.05)
    ROI_Y = int(h * 0.2)

    # Ensure ROI stays inside frame
    if ROI_Y + ROI_SIZE > h or ROI_X + ROI_SIZE > w:
        continue

    roi = frame[ROI_Y:ROI_Y + ROI_SIZE, ROI_X:ROI_X + ROI_SIZE]

    # ── Preprocess ROI ─────────────────────────────────────────────────────
    roi_resized = cv2.resize(roi, IMG_SIZE_RT)
    roi_rgb = cv2.cvtColor(roi_resized, cv2.COLOR_BGR2RGB)
    roi_normalized = roi_rgb.astype('float32') / 255.0
    roi_batch = np.expand_dims(roi_normalized, axis=0)

    # ── Predict every 3rd frame ────────────────────────────────────────────
    if frame_count % 3 == 0:
        predictions = rt_model.predict(roi_batch, verbose=0)[0]
        top_idx = np.argmax(predictions)
        confidence = predictions[top_idx]

        # 🔥 Smoothing
        recent_preds.append(top_idx)
        if len(recent_preds) > 5:
            recent_preds.pop(0)

        top_idx = max(set(recent_preds), key=recent_preds.count)
        top_label = rt_class_names[top_idx]

        top3_idx = np.argsort(predictions)[::-1][:3]

    # ── Draw ROI Box ──────────────────────────────────────────────────────
    box_color = (0, 255, 0) if confidence >= CONFIDENCE_THRESHOLD else (0, 165, 255)

    cv2.rectangle(frame,
                  (ROI_X, ROI_Y),
                  (ROI_X + ROI_SIZE, ROI_Y + ROI_SIZE),
                  box_color, 3)

    cv2.putText(frame, "Place Hand Here",
                (ROI_X, ROI_Y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, box_color, 2)

    # ── Display Prediction ────────────────────────────────────────────────
    text_x = ROI_X + ROI_SIZE + 20
    text_y = ROI_Y + 50

    if confidence >= CONFIDENCE_THRESHOLD:
        cv2.putText(frame, f"Sign: {top_label}",
                    (text_x, text_y),
                    cv2.FONT_HERSHEY_DUPLEX, 1.2, (0, 255, 0), 3)

        cv2.putText(frame, f"Conf: {confidence*100:.1f}%",
                    (text_x, text_y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    else:
        cv2.putText(frame, "Low Confidence...",
                    (text_x, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2)

    # ── Top 3 Predictions ─────────────────────────────────────────────────
    cv2.putText(frame, "Top 3:",
                (text_x, text_y + 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    for rank, idx in enumerate(top3_idx):
        text = f"{rank+1}. {rt_class_names[idx]}: {predictions[idx]*100:.1f}%"
        cv2.putText(frame, text,
                    (text_x, text_y + 120 + rank * 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)

    # ── Bottom Instructions ───────────────────────────────────────────────
    cv2.putText(frame, "Q: Quit | S: Screenshot",
                (10, h - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (150, 150, 150), 1)

    # ── Show Window ───────────────────────────────────────────────────────
    cv2.imshow("ASL Detector", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        print("👋 Quitting...")
        break
    elif key == ord('s'):
        filename = f'screenshot_{frame_count}.png'
        cv2.imwrite(filename, frame)
        print(f"📸 Screenshot saved: {filename}")

cap.release()
cv2.destroyAllWindows()
print("✅ Webcam closed.")

🎥 Webcam opened. Place hand in the GREEN BOX.
Press 'Q' to quit | Press 'S' to save screenshot


QFontDatabase: Cannot find font directory /home/kanishk/repos/sign-language-detection/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/kanishk/repos/sign-language-detection/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/kanishk/repos/sign-language-detection/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/kanishk/repos/sign-language-detection/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ fo

👋 Quitting...
✅ Webcam closed.
